In [ ]:
# knowlytix and forgeloop are installed from PyPI (pip install knowlytix forgeloop)
import os

# Ch15 — Persisting to an external store (KAL / Postgres)

A trained GMS store lives on local disk. To serve a fleet, the *graph* —
triples plus provenance plus GEODE verification — has to live in a real
database. KAL (`knowlytix.kal`, the Knowledge Adapter Layer) is the
org's backend-agnostic knowledge-graph store; `knowlytix.knowledge.rag.kal_sink`
is the bridge. This notebook converts the store's triples to `KALTriple`
records, persists them through the offline mock adapter, reads them back,
and shows the (gated) Postgres path. No GPU, no Qwen, no trained store: the
fixtures below reproduce the corpus exactly (see `data/corpus_facts.md`).

## A corpus-grounded fixture store

`store_to_kal_triples` only touches three attributes of a built store:
`store.triples` (an iterable of `(head, relation, tail)` tuples),
`store.markdown` (the source document, for provenance recovery), and
`store.store_path` (the default `source`). We stand in a tiny object with
exactly those attributes, populated from the canonical corpus facts, so the
chapter runs offline. In CI the lead swaps this for the real store via
`build_rag_store('data/annual_report.md', ...)` — the conversion code is
identical.

In [ ]:
from dataclasses import dataclass, field

# (head, relation, tail) tuples — verbatim from data/corpus_facts.md.
# Numeric tails are exact ENM values; entity tails are graph nodes.
CORPUS_TRIPLES = [
    ("cloud platform", "has_revenue", "120.0"),
    ("cloud platform", "has_headcount", "340.0"),
    ("cloud platform", "has_division", "technology"),
    ("devices", "has_revenue", "80.0"),
    ("devices", "has_division", "technology"),
    ("logistics", "has_revenue", "95.0"),
    ("logistics", "has_division", "operations"),
    ("retail", "has_revenue", "60.0"),
    ("retail", "has_division", "operations"),
    ("total", "has_revenue", "355.0"),
    ("technology", "has_region", "north america"),
    ("technology", "has_head", "dana cole"),
    ("operations", "has_region", "europe"),
    ("revenue", "has_fy2025", "355.0"),
    # in_section triples exist in the real store but are excluded by
    # store_to_kal_triples' default exclude_relations=('in_section',):
    ("cloud platform", "in_section", "segment performance"),
]

# A minimal markdown stand-in so the ProvenanceLedger can recover
# source_text for the table-cell triples (line/char spans).
FIXTURE_MD = (
    "## 1. Segment Performance\n"
    "| Segment | Division | Revenue | Headcount |\n"
    "| --- | --- | --- | --- |\n"
    "| Cloud Platform | Technology | 120.0 | 340.0 |\n"
    "| Devices | Technology | 80.0 | 210.0 |\n"
    "| Total | All | 355.0 | 1500.0 |\n"
)

@dataclass
class FixtureStore:
    """Stands in for a built RAG store (offline). Same attribute surface
    that kal_sink reads."""
    triples: list = field(default_factory=lambda: list(CORPUS_TRIPLES))
    markdown: str = FIXTURE_MD
    store_path: str = "data/gms_annual_report_store"

store = FixtureStore()
len(store.triples)

## Listing 1 — Convert the store's triples to `KALTriple`

`store_to_kal_triples` maps each `(h, r, t)` to a `KALTriple`. Two design
rules matter here:

- **Numeric tails become literals.** `('cloud platform', 'has_revenue', '120.0')`
  has a numeric tail, so it lands as `object_literal=KALLiteral(value='120.0',
  datatype='number')` — the exact ENM value rides along, byte-for-byte. Entity
  tails like `technology` become object **nodes**.
- **Provenance is recovered**, not invented. The `source_text` is pulled from
  `store.markdown` via a `ProvenanceLedger`; the `extractor` is stamped `geode`.
- **`in_section` is dropped** by the default `exclude_relations`.

In [ ]:
from knowlytix.knowledge.rag.kal_sink import store_to_kal_triples

kal_triples = store_to_kal_triples(store, extractor="geode")

print(f"converted {len(kal_triples)} triples "
      f"(from {len(store.triples)} store triples; in_section excluded)")

# A numeric (literal) triple vs an entity (node) triple.
rev = next(t for t in kal_triples
           if t.subject.normalized_name == "cloud platform"
           and t.predicate == "has_revenue")
div = next(t for t in kal_triples
           if t.subject.normalized_name == "cloud platform"
           and t.predicate == "has_division")

print("numeric tail  ->", rev.object, "| literal:", rev.object_literal)
print("entity tail   ->", div.object, "| literal:", div.object_literal)
print("provenance    ->", rev.provenance)

**What the output means.** The revenue triple carries
`object_literal=KALLiteral(value='120.0', datatype='number')` and `object=None`;
the division triple carries `object=KALNode(... name='technology')` and
`object_literal=None`. KAL's own model validator enforces exactly-one-of, so a
malformed triple can't be constructed. The exact figure was never re-parsed from
prose — it was carried through from the ENM value as a typed literal.

## Listing 2 — Persist via the offline mock adapter and read back

`MockKnowledgeAdapter` is KAL's in-memory adapter (it lives in production
source, not `tests/`). It is the offline persistence path: no Postgres, no
network. `persist_store_to_kal` is the async bridge around the async
`insert_triples`; a notebook is already running inside an event loop, so we
`await` it directly (the `persist_store_to_kal_sync` wrapper is for plain
scripts — its `asyncio.run` cannot nest inside the notebook's loop). We seed
the mock's `fixed_triples` with the converted records so a subsequent
`query_triples` reads them back — a full round trip.

In [ ]:
from knowlytix.kal import KALQuery
from knowlytix.kal.adapters.mock import MockKnowledgeAdapter
from knowlytix.knowledge.rag.kal_sink import persist_store_to_kal

# Seed the mock with what we will persist, so reads return them.
adapter = MockKnowledgeAdapter("geode-mock", fixed_triples=kal_triples)

# Write through the async bridge (top-level await; a notebook is already
# inside a running event loop, so the sync wrapper's asyncio.run would fail).
inserted = await persist_store_to_kal(adapter, store, tenant_id="northwind")
print(f"persisted {inserted} triples")

# Read them back through the adapter's async query API.
result = await adapter.query_triples(KALQuery(), tenant_id="northwind")
read_back = result.triples
print(f"read back {len(read_back)} triples; errors={result.errors}")

for t in read_back[:3]:
    tail = (t.object_literal.value if t.object_literal
            else t.object.name)
    print(f"  {t.subject.name} -- {t.predicate} --> {tail}")

**What the output means.** `inserted` equals the converted-triple count, and the
read-back set has the same length: the graph survives the round trip with its
structure intact. The mock does not persist across process restarts — that is the
Postgres adapter's job (Listing 4) — but it exercises the exact same
`KnowledgeAdapter` Protocol the real backend implements.

## Listing 3 — Stamp `confidence`, read the verification back (exercise)

GEODE verification metadata is first-class in KAL. Pass `confidence=...` to
`store_to_kal_triples` and every triple gets a `VerificationMetadata` with
`verification_status='verified'` and `verifier='geode'`. This is the
exercise's worked solution: persist with a stamped confidence, then read the
verification back off a round-tripped triple.

In [ ]:
verified_triples = store_to_kal_triples(store, extractor="geode",
                                        confidence=0.97)
v_adapter = MockKnowledgeAdapter("geode-verified",
                                 fixed_triples=verified_triples)
v_result = await v_adapter.query_triples(KALQuery())

sample = v_result.triples[0]
vm = sample.verification
print("confidence        ->", vm.confidence)
print("verification_status ->", vm.verification_status)
print("verifier          ->", vm.verifier)

# Without confidence, verification is None (the default in Listing 1).
print("unstamped verification ->", kal_triples[0].verification)

**What the output means.** `confidence=0.97` produced
`VerificationMetadata(confidence=0.97, verification_status='verified',
verifier='geode')`; the unstamped Listing-1 triples have `verification=None`.
Verification travels *with* the triple across federation — a consumer can filter
on `min_confidence` without re-running GEODE.

## Listing 4 — The Postgres path (shown, gated on a DSN env var)

The real backend is `kal_postgres_adapter`, which builds a pgvector-backed KAL
adapter through `AdapterFactory`. It requires a live database with KAL's
migrations applied, so this cell is **gated on `KAL_PG_DSN`** and is a no-op in
offline CI. The conversion + persist calls are byte-identical to the mock path —
only the adapter differs.

In [ ]:
import os
from knowlytix.knowledge.rag.kal_sink import (
    kal_postgres_adapter, persist_store_to_kal)

PG = os.environ.get("KAL_PG_DSN")  # e.g. host=...;db=...;user=...;pw=...
if PG:
    parts = dict(kv.split("=", 1) for kv in PG.split(";"))
    pg_adapter = kal_postgres_adapter(
        host=parts["host"], database=parts["db"],
        username=parts["user"], password=parts["pw"])
    n = await persist_store_to_kal(pg_adapter, store,
                                   tenant_id="northwind", confidence=0.97)
    print(f"persisted {n} triples to Postgres")
else:
    print("KAL_PG_DSN not set -- skipping live Postgres persist (offline CI)")

**What the output means.** With no DSN, the cell prints the skip notice and CI
stays offline. Pointed at a migrated database, the same `store` would persist
with pgvector-backed similarity search available on the nodes. The trained GMS
model checkpoint is *not* sent to KAL — only the graph is. See `data/corpus_facts.md`
for the full ENM table the real store would carry.

## Self-check

The chapter's claim: the converted graph round-trips through a KAL adapter with
its count, numeric literals, and provenance intact.

In [ ]:
# 1. in_section was excluded; everything else converted.
assert len(kal_triples) == len(store.triples) - 1

# 2. Round-trip count through the mock adapter matches.
assert inserted == len(kal_triples)
assert len(read_back) == len(kal_triples)

# 3. Numeric ENM value rode through as a typed literal, byte-exact.
assert rev.object is None
assert rev.object_literal.value == "120.0"
assert rev.object_literal.datatype == "number"

# 4. Entity tail became a node, not a literal.
assert div.object is not None and div.object.name == "technology"
assert div.object_literal is None

# 5. Verification stamps when confidence is supplied, None otherwise.
assert vm.confidence == 0.97 and vm.verification_status == "verified"
assert kal_triples[0].verification is None

print("Ch15 self-check passed: graph round-trips through KAL intact.")